In [50]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score, confusion_matrix
import numpy as np

feat_path   = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\Extracted_features\features_ViTTiny.xlsx"
label_path  = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\scientificProject\data\labels.csv"

feat_df = pd.read_excel(feat_path, header=None)
feat_df.rename(columns={feat_df.columns[0]: "name"}, inplace=True)
feat_df["name"] = feat_df["name"].astype(str).str.replace(".png", "", regex=False)


feat_png = feat_df["name"].astype(str) + ".jpg"

label_df = pd.read_csv(label_path)
label_df["name"] = label_df["name"].astype(str)
label_df = label_df[label_df["name"].isin(feat_png)]
feat_df['name'] = feat_df['name'] + '.jpg'
data = feat_df.merge(label_df, on="name", how="inner")

X = data.drop(columns=["name", "category", 'type', 'grade'])
y = data["category"]

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [51]:
def specificity_multiclass(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    specificity_per_class = []
    for i in range(len(cm)):
        tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
        fp = cm[:, i].sum() - cm[i, i]
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        specificity_per_class.append(spec)
    return np.mean(specificity_per_class)


model = LogisticRegression(penalty='l2', C=1, solver='liblinear', max_iter=100, multi_class='ovr')

model.fit(x_train, y_train)

y_train_pred = model.predict(x_train)
y_test_pred = model.predict(x_test)

train_acc = accuracy_score(y_train, y_train_pred)
train_bal_acc = balanced_accuracy_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred, average='weighted')
train_sens = recall_score(y_train, y_train_pred, average='weighted') 
train_spec = specificity_multiclass(y_train, y_train_pred)

test_acc = accuracy_score(y_test, y_test_pred)
test_bal_acc = balanced_accuracy_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred, average='weighted')
test_sens = recall_score(y_test, y_test_pred, average='weighted')
test_spec = specificity_multiclass(y_test, y_test_pred)

print("Train Metrics:")
print(f"Accuracy: {train_acc:.4f}")
print(f"Balanced Accuracy: {train_bal_acc:.4f}")
print(f"F1 Score: {train_f1:.4f}")
print(f"Sensitivity (Recall): {train_sens:.4f}")
print(f"Specificity: {train_spec:.4f}")

print("\nTest Metrics:")
print(f"Accuracy: {test_acc:.4f}")
print(f"Balanced Accuracy: {test_bal_acc:.4f}")
print(f"F1 Score: {test_f1:.4f}")
print(f"Sensitivity (Recall): {test_sens:.4f}")


In [52]:
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True)

plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix on Test Data')
plt.show()